In [ ]:
import os
from pathlib import Path
from typing import Final

import ipywidgets as widgets
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display
from plotly.subplots import make_subplots

import tsam

# Purpose And Scope
This notebook starts from cleaned hourly input datasets, validates the data, runs a single annual TSAM aggregation, and inspects the resulting representative-period outputs.

## Table of Contents

- [Purpose And Scope](#purpose-and-scope)
- [Data Path Configuration](#data-path-configuration)
- [Data Loading](#data-loading)
- [Snapshot Feature Handling](#snapshot-feature-handling)
- [Data Validation](#data-validation)
  - [Duplicates, NaNs, Numeric Columns & Full-Year Coverage](#duplicates-nans-numeric-columns--full-year-coverage)
  - [Capacity-Factor Range Validation](#capacity-factor-range-validation)
  - [Running Validation](#running-validation)
- [TSAM Clustering](#tsam-clustering)
- [Inspect Aggregation Outputs](#inspect-aggregation-outputs)
  - [Output Shapes](#output-shapes)
  - [Raw TSAM Result Outputs](#raw-tsam-result-outputs)
  - [Accuracy Tables](#accuracy-tables)
  - [Extremes Comparison](#extremes-comparison)
- [Diagnostic Visualizations](#diagnostic-visualizations)
  - [Plotting setup](#plotting-setup)
  - [Cluster Representatives](#cluster-representatives)
  - [Cluster Members](#cluster-members)
  - [Cluster Weights](#cluster-weights)
  - [Cluster Accuracy](#cluster-accuracy)
  - [Full-Resolution Comparison](#full-resolution-comparison)
  - [Residuals](#residuals)
  - [Calendar Heatmap Of Original Days By Assigned Cluster](#calendar-heatmap-of-original-days-by-assigned-cluster)
  - [Cluster Weight Heatmap](#cluster-weight-heatmap)
- [Saving Results](#saving-results)

# Data Path Configuration

In [60]:
# Notebook working directory. In this project, running from src/ makes ../data resolve to data/.
cur_location: Path = Path().resolve()
data_location: Path = cur_location / "../data"

# Data Loading


Specify the CSV separator through `sep`. The input datasets use both comma-separated and semicolon-separated formats.

In [61]:
demand_EU = pd.read_csv(data_location / "Demand_ENTSO_E.csv", sep=";")
demand_EU.head()

,snapshot,AL,AT,BA,BE,BG,CH,CZ,DE,DK,...,NL,NO,PL,PT,RO,RS,SE,SI,SK,UK
0,01.01.2025 00:00,459,3871,609,6691,2374,4603,4029,28833,2606,...,9796,11525,11852,3591,3645,2578,9701,728,1823,18285
1,01.01.2025 01:00,428,3717,571,6327,2292,4711,3989,27781,2511,...,9535,11586,11284,3480,3532,2474,9551,699,1780,18154
2,01.01.2025 02:00,396,3522,539,6128,2237,4861,3885,26917,2426,...,9266,11540,10983,3289,3467,2316,9377,666,1733,16545
3,01.01.2025 03:00,376,3488,523,5982,2206,4810,3855,27016,2400,...,9013,11517,10827,3126,3438,2175,9417,648,1729,14918
4,01.01.2025 04:00,367,3598,526,5967,2194,4725,3865,26870,2405,...,9040,11602,10754,3007,3428,2059,9560,664,1724,14200


In [62]:
onwind_cf = pd.read_csv(data_location / "onwind_capacity_factors.csv", sep=",")
onwind_cf.head()

,snapshot,AL_onwind_2025,AT_onwind_2025,BA_onwind_2025,BE_onwind_2025,BG_onwind_2025,CH_onwind_2025,CZ_onwind_2025,DE_onwind_2025,DK_onwind_2025,...,NO_onwind_2025,PL_onwind_2025,PT_onwind_2025,RO_onwind_2025,RS_onwind_2025,SE_onwind_2025,SI_onwind_2025,SK_onwind_2025,UA_onwind_2025,XK_onwind_2025
0,01.01.2025 00:00,0.0,0.026443,0.0,0.968038,0.027777,0.030767,0.419450,0.716840,1.000000,...,0.131214,0.873780,0.116221,0.051165,0.0,0.297268,0.013709,0.114870,0.346401,0.0
1,01.01.2025 01:00,0.0,0.029090,0.0,0.975868,0.027310,0.030301,0.426459,0.728613,0.999992,...,0.109802,0.897004,0.109594,0.054343,0.0,0.286711,0.015315,0.114240,0.345689,0.0
2,01.01.2025 02:00,0.0,0.033465,0.0,0.983856,0.028493,0.029116,0.443365,0.743509,0.999972,...,0.092031,0.911984,0.105639,0.062091,0.0,0.278833,0.019344,0.110237,0.332158,0.0
3,01.01.2025 03:00,0.0,0.034560,0.0,0.988063,0.030633,0.029512,0.467903,0.760476,0.999996,...,0.078412,0.923292,0.107288,0.067657,0.0,0.271188,0.023585,0.110385,0.326958,0.0
4,01.01.2025 04:00,0.0,0.036894,0.0,0.987477,0.034158,0.031685,0.495288,0.771249,1.000000,...,0.071422,0.927705,0.103116,0.072680,0.0,0.259012,0.025262,0.112671,0.322901,0.0


In [63]:
hydro_inflow = pd.read_csv(
    data_location / "hydro_inflow_scaled_2025.csv", sep=";"
)
hydro_inflow.head()

,snapshot,AT_hydro_2025,BA_hydro_2025,BE_hydro_2025,BG_hydro_2025,CH_hydro_2025,CZ_hydro_2025,DE_hydro_2025,ES_hydro_2025,FI_hydro_2025,...,RO_hydro_2025.1,RS_hydro_2025,RS_hydro_2025.1,RS_hydro_2025.2,SI_hydro_2025,UA_hydro_2025,UK_hydro_2025,XK_hydro_2025,SE_hydro_2025,AL_hydro_2025
0,01.01.2025 00:00,230,83,2,49,411,39,54,922,420,...,56,2,2,8,135,556,43,3,4570,155
1,01.01.2025 01:00,229,83,2,49,411,40,54,922,420,...,56,2,2,8,135,555,43,3,4569,152
2,01.01.2025 02:00,229,83,2,49,412,41,54,922,421,...,56,2,2,8,135,555,43,3,4562,151
3,01.01.2025 03:00,228,83,2,49,412,41,54,922,422,...,56,2,2,8,135,555,43,3,4555,151
4,01.01.2025 04:00,228,83,2,49,412,41,54,922,423,...,56,2,2,8,135,555,43,3,4545,150


In [64]:
ror_cf = pd.read_csv(data_location / "ror_capacity_factors.csv", sep=",")
ror_cf.head()

,snapshot,AL_ror_2025,AT_ror_2025,BA_ror_2025,BE_ror_2025,BG_ror_2025,CH_ror_2025,CZ_ror_2025,DE_ror_2025,EE_ror_2025,...,NO_ror_2025,PL_ror_2025,PT_ror_2025,RO_ror_2025,RS_ror_2025,SE_ror_2025,SI_ror_2025,SK_ror_2025,UK_ror_2025,XK_ror_2025
0,01.01.2025 00:00,0.06198,0.304260,0.113561,0.687213,0.132907,0.266242,0.128353,0.596077,0.0,...,0.592028,0.503166,0.140826,0.371941,0.428571,0.35421,0.090094,0.232892,0.19516,0.07694
1,01.01.2025 01:00,0.06074,0.300330,0.149166,0.687989,0.132323,0.264119,0.406737,0.591670,0.0,...,0.585834,0.503452,0.062204,0.369386,0.411529,0.35409,0.087915,0.231347,0.19512,0.07694
2,01.01.2025 02:00,0.06031,0.300783,0.155915,0.626787,0.129821,0.262051,0.430766,0.584802,0.0,...,0.578365,0.502625,0.059455,0.363477,0.338847,0.35360,0.087664,0.233002,0.19510,0.07682
3,01.01.2025 03:00,0.06016,0.306833,0.111967,0.609455,0.130377,0.260064,0.438508,0.577704,0.0,...,0.576665,0.502492,0.044515,0.359085,0.249123,0.35307,0.087212,0.230574,0.19506,0.07675
4,01.01.2025 04:00,0.05997,0.305347,0.111861,0.587503,0.135743,0.258158,0.438407,0.575100,0.0,...,0.580272,0.502750,0.096728,0.359085,0.246115,0.35229,0.087112,0.230022,0.19502,0.07684


In [65]:
solar_cf = pd.read_csv(data_location / "solar_capacity_factors.csv", sep=";")
solar_cf.head()

,snapshot,AL_solar_2025,AT_solar_2025,BA_solar_2025,BE_solar_2025,BG_solar_2025,CH_solar_2025,CZ_solar_2025,DE_solar_2025,DK_solar_2025,...,NO_solar_2025,PL_solar_2025,PT_solar_2025,RO_solar_2025,RS_solar_2025,SE_solar_2025,SI_solar_2025,SK_solar_2025,UA_solar_2025,XK_solar_2025
0,01.01.2025 00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,01.01.2025 01:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,01.01.2025 02:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,01.01.2025 03:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,01.01.2025 04:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


This workflow uses Germany-specific input data for the demonstration case.

In [66]:
# Slice Germany-only columns for the walkthrough.
demand_DE = demand_EU[["snapshot", "DE"]]
solar_cf_DE = solar_cf[["snapshot", "DE_solar_2025"]]
onwind_cf_DE = onwind_cf[["snapshot", "DE_onwind_2025"]]
ror_cf_DE = ror_cf[["snapshot", "DE_ror_2025"]]
hydro_inflow_DE = hydro_inflow[["snapshot", "DE_hydro_2025"]]

# Name/data pairs used by the validation loop below.
data: list[tuple[str, pd.DataFrame]] = [
    ("demand_DE", demand_DE),
    ("solar_cf_DE", solar_cf_DE),
    ("onwind_cf_DE", onwind_cf_DE),
    ("ror_cf_DE", ror_cf_DE),
    ("hydro_inflow_DE", hydro_inflow_DE),
]

# Snapshot Feature Handling
TSAM requires snapshots to be stored as the DataFrame index, and that index must use a datetime dtype.

In [67]:
def set_new_index(
    df: pd.DataFrame, new_index_col: str = "snapshot"
) -> pd.DataFrame:
    """Set ``new_index_col`` as the DataFrame index in-place and return ``df``.

    The notebook keeps separate variables like ``demand_DE`` and also stores the same
    DataFrame objects in ``data``. Mutating in-place keeps both references aligned.
    """
    if new_index_col not in df.columns:
        raise KeyError(
            f"{new_index_col!r} is not a column in the provided DataFrame"
        )

    df.set_index(new_index_col, drop=True, inplace=True)
    return df


# Set the snapshot column as the index for every source table.
data = [(df_name, set_new_index(df, "snapshot")) for df_name, df in data]

print("Index columns were swapped successfully!")

Index columns were swapped successfully!


Convert each snapshot index to datetime. The source timestamps use day-first dates; therefore, `dayfirst=True` is required.

In [68]:
# Converting index to datetime type
for df_name, df in data:
    df.index = pd.to_datetime(df.index, dayfirst=True)

print("Successfully converted indexes into datetime dtype!")

Successfully converted indexes into datetime dtype!


# Data Validation
This section validates the input data before aggregation. Failed checks raise explicit errors and identify the failing condition.

The validation step reduces the risk of silent TSAM failures or misleading representative periods.

Checks performed:

- no duplicate timestamps
- no missing values
- numeric columns only
- complete hourly year coverage
- capacity-factor columns remain within the expected `0.0-1.0` range

## Duplicates, NaNs, Numeric Columns & Full-Year Coverage
This validator checks the main TSAM input requirements: datetime index, complete hourly coverage, no duplicate timestamps, no missing values, and numeric columns only.

In [69]:
def validate_df(
    df: pd.DataFrame,
    name: str,
    year_to_check: int = 2025,
    period: str = "h",
) -> None:
    """Validate that a DataFrame is ready for hourly TSAM aggregation.

    TSAM expects a regular DatetimeIndex, numeric values, no gaps, no duplicates,
    and no NaNs. This function fails fast before aggregation can silently produce
    misleading representative periods.
    """
    expected_index = pd.date_range(
        start=f"{year_to_check}-01-01 00:00",
        end=f"{year_to_check}-12-31 23:00",
        freq=period,
    )

    if not isinstance(df.index, pd.DatetimeIndex):
        raise TypeError(f"{name}: index must be a DatetimeIndex")

    if len(df) != len(expected_index):
        raise ValueError(
            f"{name}: expected {len(expected_index)} rows, got {len(df)}"
        )

    if df.index.has_duplicates:
        raise ValueError(f"{name}: index has duplicate timestamps")

    if not df.index.equals(expected_index):
        missing = expected_index.difference(df.index)
        extra = df.index.difference(expected_index)
        raise ValueError(
            f"{name}: index does not exactly match hourly {year_to_check}. "
            f"Missing examples: {missing[:5].tolist()}. "
            f"Extra examples: {extra[:5].tolist()}."
        )

    if df.isna().any().any():
        cols = df.columns[df.isna().any()].tolist()
        raise ValueError(f"{name}: contains NaNs in columns {cols}")

    non_numeric_cols = df.select_dtypes(exclude="number").columns.tolist()
    if non_numeric_cols:
        raise TypeError(
            f"{name}: non-numeric columns found: {non_numeric_cols}"
        )


## Capacity-Factor Range Validation
Capacity-factor features are expected to remain between `0.0` and `1.0`. This validator raises an error if any capacity-factor value falls outside that interval.

In [70]:
def validate_cf_range(df: pd.DataFrame, name: str) -> None:
    """Validate that capacity-factor columns stay inside the physical 0..1 range."""
    is_in_range = df.ge(0.0) & df.le(1.0)

    if not is_in_range.all().all():
        invalid_cols = df.columns[~is_in_range.all()].tolist()
        raise ValueError(
            f"{name}: values outside the 0.0-1.0 CF range in {invalid_cols}"
        )


## Running Validation
Run all validation checks. A failed check raises an error that identifies the violated condition.

In [71]:
capacity_factor_datasets: Final[set[str]] = {
    "solar_cf_DE",
    "onwind_cf_DE",
    "ror_cf_DE",
}

for name, df in data:
    validate_df(df, name, year_to_check=2025, period="h")

    if name in capacity_factor_datasets:
        validate_cf_range(df, name)

print("Data validation has been successful!")

Data validation has been successful!


TSAM expects one dataset for the region being aggregated. The Germany-specific features are joined into a single DataFrame.

In [72]:
data_DE: pd.DataFrame = demand_DE.join(
    [solar_cf_DE, onwind_cf_DE, ror_cf_DE, hydro_inflow_DE]
).rename(columns={"DE": "DE_demand_2025"})
FEATURE_COLUMNS = data_DE.columns

data_DE.head()

,DE_demand_2025,DE_solar_2025,DE_onwind_2025,DE_ror_2025,DE_hydro_2025
snapshot,,,,,
2025-01-01 00:00:00,28833,0.0,0.716840,0.596077,54
2025-01-01 01:00:00,27781,0.0,0.728613,0.591670,54
2025-01-01 02:00:00,26917,0.0,0.743509,0.584802,54
2025-01-01 03:00:00,27016,0.0,0.760476,0.577704,54
2025-01-01 04:00:00,26870,0.0,0.771249,0.575100,54


# TSAM Clustering
This baseline notebook performs one annual TSAM aggregation using the configured cluster count, 24-hour periods, hierarchical clustering, and medoid representation.

[API reference](https://tsam.readthedocs.io/en/latest/api/tsam/api/#tsam.api.aggregate)

In [73]:
BASELINE_CLUSTER_CONFIG = tsam.ClusterConfig(
    method="hierarchical",
    representation="medoid",
)
PERIOD_DURATION_HOURS: Final[int] = 24

aggregation_result = tsam.aggregate(
    data=data_DE.loc[:, FEATURE_COLUMNS],
    n_clusters=6,  # Number of representative days.
    period_duration=PERIOD_DURATION_HOURS,
    preserve_column_means=True,  # Rescale representatives to preserve annual column sums.
    cluster=BASELINE_CLUSTER_CONFIG,
)

Important `tsam.aggregate` options used in this baseline:

- `method`: determines how original periods are grouped
- `representation`: determines how each cluster is represented
- `preserve_column_means`: rescales typical periods so each column's weighted mean matches the original data's mean

## Inspect Aggregation Outputs
[API reference](https://tsam.readthedocs.io/en/latest/api/tsam/result/#tsam.result.AggregationResult)

This section inspects the main objects returned by TSAM.

### Output Shapes
Compare the data shape before and after aggregation.

In [74]:
print(
    f"Num of rows and num of cols in the original full-year dataset: {data_DE.shape}"
)
print(
    f"Num of rows and num of cols in the representative output: {aggregation_result.cluster_representatives.shape}"
)

Num of rows and num of cols in the original full-year dataset: (8760, 5)
Num of rows and num of cols in the representative output: (144, 5)


The number of features remains unchanged. The reduced time dimension is determined by the configured number of representative periods multiplied by the 24-hour period duration.

Inspect the number of original periods, representative periods, and timesteps per representative period.

In [75]:
n_representative_periods, n_timesteps_per_period = (
    aggregation_result.n_clusters,
    aggregation_result.n_timesteps_per_period,
)

print(f"Num of original periods: {data_DE.shape[0]}")
print(f"Num of representative periods: {n_representative_periods}")
print(
    f"Number of timesteps per representative period: {n_timesteps_per_period}"
)

Num of original periods: 8760
Num of representative periods: 6
Number of timesteps per representative period: 24


### Raw TSAM Result Outputs
Inspect the main aggregation outputs exposed by TSAM.

#### TSAM Cluster Representatives Output

In [76]:
aggregation_result.cluster_representatives.head()

DE_demand_2025  DE_hydro_2025  DE_onwind_2025  DE_ror_2025  \
  timestep                                                               
0 0           32410.062815     187.109133        0.125963     0.618568   
  1           31702.972592     187.109133        0.131710     0.615149   
  2           31651.582455     188.054713        0.137004     0.611909   
  3           31458.393606     188.054713        0.144796     0.609348   
  4           31440.311891     189.000293        0.150914     0.607840   

            DE_solar_2025  
  timestep                 
0 0                   0.0  
  1                   0.0  
  2                   0.0  
  3                   0.0  
  4                   0.0

The index has the shape `(cluster number, hour of day)`. With `representation="medoid"`, each representative profile is based on a real day selected from its cluster.

#### Cluster Assignments

Inspect the output shape before reviewing the assignments.

In [77]:
aggregation_result.cluster_assignments.shape[0]

365

Inspect the assignment values.

In [78]:
aggregation_result.cluster_assignments

array([5, 1, 1, 1, 4, 5, 5, 1, 1, 1, 1, 2, 2, 1, 1, 1, 1, 1, 2, 2, 1, 1,
       1, 1, 1, 1, 5, 1, 1, 1, 1, 1, 2, 2, 1, 1, 1, 1, 2, 2, 4, 1, 1, 1,
       1, 1, 2, 2, 2, 2, 2, 2, 1, 2, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3,
       3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 3, 3, 3, 3, 3, 3, 3, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 2,
       2, 2, 2, 0, 0, 2, 2, 2, 2, 2, 0, 0, 2, 2, 2, 2, 2, 0, 0, 2, 2, 2,
       2, 2, 0, 0, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 0, 0, 2, 2, 2, 2, 2, 0, 0, 2, 2, 2, 2, 2, 0, 0, 2, 2, 2, 2, 2,
       0, 0, 2, 2, 2, 2, 2, 0, 0, 2, 2, 2, 2, 2, 0, 5, 5, 2, 2, 2, 2, 4,
       4, 2, 2, 2, 2, 2, 0, 0, 2, 2, 2, 4, 5, 5, 5, 2, 2, 2, 1, 1, 1, 0,
       1, 1, 1, 1, 2, 4, 4, 1, 1, 5, 5, 5, 5, 5, 5,

This 365-element array identifies the cluster assigned to each original day. For example, the first day belongs to cluster `5`, while the second, third, and fourth days belong to cluster `1`.

#### Assignments

In [79]:
aggregation_result.assignments

,period_idx,timestep_idx,cluster_idx
snapshot,,,
2025-01-01 00:00:00,0,0,5
2025-01-01 01:00:00,0,1,5
2025-01-01 02:00:00,0,2,5
2025-01-01 03:00:00,0,3,5
2025-01-01 04:00:00,0,4,5
...,...,...,...
2025-12-31 19:00:00,364,19,4
2025-12-31 20:00:00,364,20,4
2025-12-31 21:00:00,364,21,4


This is the same assignment data with additional indexing columns.

Columns:

- `period_idx`: chronological period number. With `period_duration=24`, one period equals one day.
- `timestep_idx`: hour inside the period. With 24-hour periods, it ranges from `0` to `23`.
- `cluster_idx`: cluster assigned to that period.

#### TSAM Cluster Weights Output

In [80]:
aggregation_result.cluster_weights

{np.int64(0): np.int64(96),
 np.int64(1): np.int64(78),
 np.int64(2): np.int64(101),
 np.int64(3): np.int64(45),
 np.int64(4): np.int64(29),
 np.int64(5): np.int64(16)}

Cluster weights show how many original days each representative day represents. For example, if cluster `0` has weight `96`, its representative day replaces 96 original days.

#### Reconstructed Data

In [81]:
aggregation_result.reconstructed.head()

,DE_demand_2025,DE_hydro_2025,DE_onwind_2025,DE_ror_2025,DE_solar_2025
snapshot,,,,,
2025-01-01 00:00:00,32854.492336,77.421827,0.580326,0.629560,0.0
2025-01-01 01:00:00,31891.403096,78.367407,0.589765,0.627519,0.0
2025-01-01 02:00:00,31640.162425,78.367407,0.641840,0.624338,0.0
2025-01-01 03:00:00,31805.752867,78.367407,0.720882,0.623942,0.0
2025-01-01 04:00:00,31439.360221,78.367407,0.777625,0.625706,0.0


The reconstructed dataset has the original 8760-hour index. Each original day is replaced by the representative day assigned to its cluster.

#### TSAM Residuals Output

In [82]:
aggregation_result.residuals.head()

,DE_demand_2025,DE_hydro_2025,DE_onwind_2025,DE_ror_2025,DE_solar_2025
snapshot,,,,,
2025-01-01 00:00:00,-4021.492336,-23.421827,0.136514,-0.033483,0.0
2025-01-01 01:00:00,-4110.403096,-24.367407,0.138848,-0.035849,0.0
2025-01-01 02:00:00,-4723.162425,-24.367407,0.101669,-0.039536,0.0
2025-01-01 03:00:00,-4789.752867,-24.367407,0.039594,-0.046238,0.0
2025-01-01 04:00:00,-4569.360221,-24.367407,-0.006376,-0.050606,0.0


Residuals are pointwise errors between the original full-resolution data and the reconstructed data.

### Accuracy Tables
[API reference](https://tsam.readthedocs.io/en/latest/api/tsam/result/#tsam.result.AccuracyMetrics)

In [83]:
aggregation_result.accuracy

AccuracyMetrics(
  rmse=0.1287 (weighted),
  mae=0.0918 (weighted),
  rmse_duration=0.0596 (weighted)
)

Accuracy metrics compare the aggregated/reconstructed data against the original data:

- `RMSE`: penalizes large errors more heavily
- `MAE`: treats errors more evenly
- `RMSE_duration`: measures error in the duration curve, where value distribution matters more than timing

Inspect the errors per feature column.

In [84]:
aggregation_result.accuracy.rmse

DE_demand_2025    0.136296
DE_hydro_2025     0.099394
DE_onwind_2025    0.144828
DE_ror_2025       0.146049
DE_solar_2025     0.109796
Name: RMSE, dtype: float64

In [85]:
aggregation_result.accuracy.mae

DE_demand_2025    0.105167
DE_hydro_2025     0.068091
DE_onwind_2025    0.110071
DE_ror_2025       0.116609
DE_solar_2025     0.059206
Name: MAE, dtype: float64

In [86]:
aggregation_result.accuracy.rmse_duration

DE_demand_2025    0.052320
DE_hydro_2025     0.064524
DE_onwind_2025    0.072079
DE_ror_2025       0.070510
DE_solar_2025     0.026313
Name: RMSE_duration, dtype: float64

#### Extremes Comparison
Compare annual minimum and maximum values to assess how well the reconstructed data preserves extreme values.

In [ ]:
original_features: pd.DataFrame = data_DE.loc[:, FEATURE_COLUMNS]
reconstructed_features: pd.DataFrame = aggregation_result.reconstructed.loc[
    :, FEATURE_COLUMNS
]

Calculate minimum and maximum errors.

In [88]:
def build_extreme_value_comparison(
    original_features: pd.DataFrame,
    reconstructed_features: pd.DataFrame,
) -> pd.DataFrame:
    """Compare annual min/max values between original and reconstructed series."""
    comparison = pd.DataFrame(
        {
            "original_min": original_features.min(axis=0),
            "reconstructed_min": reconstructed_features.min(axis=0),
            "original_max": original_features.max(axis=0),
            "reconstructed_max": reconstructed_features.max(axis=0),
        }
    )

    comparison["min_error"] = (
        comparison["reconstructed_min"] - comparison["original_min"]
    )
    comparison["max_error"] = (
        comparison["reconstructed_max"] - comparison["original_max"]
    )
    comparison["max_error_%"] = (
        comparison["max_error"].div(comparison["original_max"]) * 100
    )

    return comparison


extreme_value_comparison = build_extreme_value_comparison(
    original_features, reconstructed_features
)
extreme_value_comparison


,original_min,reconstructed_min,original_max,reconstructed_max,min_error,max_error,max_error_%
DE_demand_2025,22625.000000,28557.705854,53574.000000,48305.793627,5932.705854,-5268.206373,-9.833513
DE_solar_2025,0.000000,0.000000,0.766724,0.607391,0.000000,-0.159333,-20.780984
DE_onwind_2025,0.000000,0.067996,0.914319,0.914319,0.067996,0.000000,0.000000
DE_ror_2025,0.344618,0.465641,1.000000,0.791893,0.121023,-0.208107,-20.810735
DE_hydro_2025,50.000000,77.421827,463.000000,318.544784,27.421827,-144.455216,-31.199831


# Diagnostic Visualizations
[API reference](https://tsam.readthedocs.io/en/latest/api/tsam/plot/#tsamplot)

This section presents TSAM diagnostic plots for representative profiles, cluster membership, reconstruction quality, and calendar assignments.

In [89]:
# Setting visual theme here for persistence
plotly_theme: Final[str] = "ggplot2"

The dataset contains features on different scales: capacity factors are in the `0-1` range, while demand is much larger. Feature dropdowns separate these scale groups for inspection.

In [90]:
FEATURE_GROUPS: Final[dict[str, list[str]]] = {
    "capacity_factors": ["DE_onwind_2025", "DE_ror_2025", "DE_solar_2025"],
    "demand": ["DE_demand_2025"],
    "hydro": ["DE_hydro_2025"],
}

## Plotting setup
These helper functions keep the feature dropdown, figure size, and Plotly styling consistent across TSAM charts.

In [ ]:
def make_feature_dropdown() -> widgets.Dropdown:
    """Create a fresh feature-group selector for one chart."""
    return widgets.Dropdown(
        options=FEATURE_GROUPS,
        description="Features:",
        layout=widgets.Layout(width="320px"),
    )


def style_tsam_figure(fig: go.Figure, title: str) -> go.Figure:
    """Apply notebook-friendly sizing to TSAM figures."""
    fig.update_layout(
        template=plotly_theme,
        title={
            "text": title,
            "x": 0.5,
            "xanchor": "center",
            "yanchor": "top",
            "font_size": 20,
        },
        autosize=True,
        width=None,
        height=720,
        margin={"l": 40, "r": 30, "t": 120, "b": 70},
        legend={
            "yanchor": "top",
            "y": 1,
            "xanchor": "left",
            "x": 1.02,
        },
    )
    # TSAM uses Plotly facet annotations for multi-column plots.
    fig.update_annotations(font_size=13)
    return fig


def display_feature_plot(plot_function) -> None:
    """Display one feature-controlled TSAM plot."""
    feature_dropdown = make_feature_dropdown()
    output = widgets.interactive_output(
        plot_function,
        {"feature_columns": feature_dropdown},
    )
    display(widgets.VBox([widgets.HBox([feature_dropdown]), output]))

## Cluster Representatives
Representative daily profiles selected by TSAM. This plot shows the typical days that replace original days.

In [ ]:
def show_cluster_representatives(feature_columns: list[str]) -> None:
    """Plot representative daily profiles for the selected feature set."""
    fig = aggregation_result.plot.cluster_representatives(
        columns=feature_columns,
        title="",
    )
    fig = style_tsam_figure(fig, title="Cluster representative profiles")
    fig.show(config={"responsive": True})


display_feature_plot(show_cluster_representatives)

## Cluster Members
Original member days are grouped by cluster, with the selected representative highlighted. The TSAM figure retains its internal cluster slider.

In [93]:
def show_cluster_members(feature_columns: list[str]) -> None:
    """Plot original member days and highlighted representatives by cluster."""
    fig = aggregation_result.plot.cluster_members(
        columns=feature_columns,
        slider="cluster",
        title="",
    )
    # Cluster selection stays inside the TSAM/Plotly figure slider.
    fig = style_tsam_figure(fig, title="Cluster members")
    fig.show(config={"responsive": True})


display_feature_plot(show_cluster_members)

## Cluster Weights
This chart shows each cluster weight: how many original days each representative day replaces.

In [94]:
fig = aggregation_result.plot.cluster_weights(title="")
fig = style_tsam_figure(fig, title="Cluster weights")
fig.show(config={"responsive": True})

## Cluster Accuracy
This chart shows accuracy metrics for the reduced/reconstructed data against the original full-resolution data.

In [95]:
fig = aggregation_result.plot.accuracy(title="")
fig = style_tsam_figure(fig, title="Cluster accuracy")
fig.show(config={"responsive": True})

## Full-Resolution Comparison
This chart compares original and reconstructed time series for the selected feature set.

In [96]:
def show_full_resolution_comparison(feature_columns: list[str]) -> None:
    """Plot original versus reconstructed time series for selected features."""
    fig = aggregation_result.plot.compare(columns=feature_columns, title="")
    fig = style_tsam_figure(fig, title="Original vs reconstructed")
    fig.show(config={"responsive": True})


display_feature_plot(show_full_resolution_comparison)

## Residuals
This chart shows pointwise reconstruction error for the selected feature set.

In [ ]:
def show_cluster_residuals(feature_columns: list[str]) -> None:
    """Plot pointwise reconstruction residuals for selected features."""
    fig = aggregation_result.plot.residuals(columns=feature_columns, title="")
    fig = style_tsam_figure(fig, title="Residuals")
    fig.show(config={"responsive": True})


display_feature_plot(show_cluster_residuals)

Calendar diagnostics show how original days were assigned to clusters. TSAM does not provide this calendar view directly, so it is constructed from the cluster assignment output.

## Calendar Heatmap Of Original Days By Assigned Cluster
Each day is colored by its assigned cluster.

The following cells prepare the calendar data and plotting helpers. The helper functions are separated to make the plotting pipeline auditable and reusable for alternative clustering configurations.

### Calendar Data Preparation Helpers

In [ ]:
ASSIGNMENT_START_DATE: Final[str] = "2025-01-01"


def build_calendar_assignments(
    assignments, start_date: str = ASSIGNMENT_START_DATE
) -> pd.DataFrame:
    """Return one row per original period with its assigned TSAM cluster."""
    assignment_dates = pd.date_range(
        start=start_date, periods=len(assignments), freq="D"
    )
    return pd.DataFrame(
        data=assignments, index=assignment_dates, columns=["cluster"]
    )


def split_assignments_by_month(
    calendar_assignments: pd.DataFrame,
) -> list[pd.DataFrame]:
    """Split the day-level assignment table into one DataFrame per month."""
    return [
        calendar_assignments[calendar_assignments.index.month == month]
        for month in range(1, 13)
    ]


def add_calendar_coordinates(month: pd.DataFrame) -> pd.DataFrame:
    """Add day, weekday, and calendar row coordinates for one month."""
    month = month.copy()
    month["day"] = month.index.day
    month["weekday"] = month.index.day_name()
    month["weekday_num"] = month.index.weekday

    first_weekday = month.index.min().weekday()
    month["week_row"] = (month["day"] + first_weekday - 1) // 7
    return month


def build_calendar_grid(month: pd.DataFrame) -> pd.DataFrame:
    """Pivot one prepared month into a week-row x weekday cluster grid."""
    return month.pivot(
        index="week_row",
        columns="weekday_num",
        values="cluster",
    )


def prep_month_for_plotting(
    month: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Return month-level plotting data and the matching calendar grid."""
    prepared_month = add_calendar_coordinates(month)
    return prepared_month, build_calendar_grid(prepared_month)


Prepare one day-level assignment table and split it into month-level plotting inputs.

In [99]:
calendar_assignments: pd.DataFrame = build_calendar_assignments(
    aggregation_result.cluster_assignments
)
monthly_calendar_assignments: list[pd.DataFrame] = split_assignments_by_month(
    calendar_assignments
)
processed_monthly_assignments: list[tuple[pd.DataFrame, pd.DataFrame]] = [
    prep_month_for_plotting(month) for month in monthly_calendar_assignments
]

### Calendar Plotting Constants

In [100]:
CALENDAR_ROWS: Final[int] = 4
CALENDAR_COLS: Final[int] = 3
WEEKDAY_NUMBERS: Final[list[int]] = list(range(7))
WEEKEND_WEEKDAY_NUMBERS: Final[set[int]] = {5, 6}
WEEKDAY_LABELS: Final[list[str]] = ["Mo", "Tu", "We", "Th", "Fr", "Sa", "Su"]
MONTH_TITLES: Final[list[str]] = [
    "January",
    "February",
    "March",
    "April",
    "May",
    "June",
    "July",
    "August",
    "September",
    "October",
    "November",
    "December",
]
CLUSTER_PALETTE: Final[list[str]] = (
    px.colors.qualitative.Plotly
    + px.colors.qualitative.Set3
    + px.colors.qualitative.Dark24
)

### Calendar Color Scale Helpers

In [101]:
def get_cluster_ids(day_clusters: pd.DataFrame) -> list[int]:
    """Return cluster IDs in display order."""
    return [
        int(cluster_id)
        for cluster_id in sorted(day_clusters["cluster"].unique())
    ]


def build_discrete_cluster_colorscale(
    cluster_ids: list[int],
) -> tuple[list[tuple[float, str]], float, float]:
    """Build a Plotly colorscale where each integer cluster has one color."""
    zmin = min(cluster_ids) - 0.5
    zmax = max(cluster_ids) + 0.5
    cluster_colors = {
        cluster_id: CLUSTER_PALETTE[i % len(CLUSTER_PALETTE)]
        for i, cluster_id in enumerate(cluster_ids)
    }

    colorscale: list[tuple[float, str]] = []
    for cluster_id in cluster_ids:
        color = cluster_colors[cluster_id]
        colorscale.append(((cluster_id - 0.5 - zmin) / (zmax - zmin), color))
        colorscale.append(((cluster_id + 0.5 - zmin) / (zmax - zmin), color))

    return colorscale, zmin, zmax


### Single-Month Heatmap Helpers

In [102]:
def build_day_text_grid(month_data: pd.DataFrame) -> pd.DataFrame:
    """Return day numbers formatted for display inside calendar cells."""
    day_number_grid = month_data.pivot(
        index="week_row",
        columns="weekday_num",
        values="day",
    ).reindex(columns=WEEKDAY_NUMBERS)
    return day_number_grid.map(
        lambda value: "" if pd.isna(value) else str(int(value))
    )


def build_weekday_hover_labels(calendar_grid: pd.DataFrame) -> pd.DataFrame:
    """Return weekday labels aligned to a calendar grid for hover text."""
    return pd.DataFrame(
        [WEEKDAY_LABELS] * len(calendar_grid),
        index=calendar_grid.index,
        columns=WEEKDAY_NUMBERS,
    )


def plot_calendar_month(
    month_data: pd.DataFrame,
    month_grid: pd.DataFrame,
    cluster_ids: list[int],
) -> go.Heatmap:
    """Build one month-calendar heatmap trace colored by assigned cluster."""
    calendar_grid = month_grid.reindex(columns=WEEKDAY_NUMBERS)
    colorscale, zmin, zmax = build_discrete_cluster_colorscale(cluster_ids)

    return go.Heatmap(
        z=calendar_grid.to_numpy(),
        x=list(WEEKDAY_NUMBERS),
        y=calendar_grid.index,
        text=build_day_text_grid(month_data).to_numpy(),
        customdata=build_weekday_hover_labels(calendar_grid).to_numpy(),
        texttemplate="%{text}",
        textfont={"size": 13, "color": "#333"},
        colorscale=colorscale,
        zmin=zmin,
        zmax=zmax,
        hovertemplate="Day %{text}<br>%{customdata}<br>Cluster %{z}<extra></extra>",
        xgap=2,
        ygap=2,
        colorbar={
            "title": "Cluster",
            "tickmode": "array",
            "tickvals": cluster_ids,
            "ticks": "",
        },
        showscale=False,
    )


Build one heatmap trace per month.

In [103]:
cluster_ids: list[int] = get_cluster_ids(calendar_assignments)
calendar_heatmaps: list[go.Heatmap] = [
    plot_calendar_month(month_data, month_grid, cluster_ids)
    for month_data, month_grid in processed_monthly_assignments
]


### Full-Year Calendar Assembly Helpers

In [ ]:
def get_subplot_positions(
    rows: int = CALENDAR_ROWS, cols: int = CALENDAR_COLS
) -> list[tuple[int, int]]:
    """Return Plotly subplot positions in row-major order."""
    return [
        (row, col) for row in range(1, rows + 1) for col in range(1, cols + 1)
    ]


def create_calendar_subplot_figure(
    heatmaps: list[go.Heatmap],
) -> tuple[go.Figure, list[tuple[int, int]]]:
    """Create the 12-month subplot figure and add monthly heatmap traces."""
    fig = make_subplots(
        rows=CALENDAR_ROWS, cols=CALENDAR_COLS, subplot_titles=MONTH_TITLES
    )
    subplot_positions = get_subplot_positions()
    fig.add_traces(
        data=heatmaps,
        rows=[row for row, _ in subplot_positions],
        cols=[col for _, col in subplot_positions],
    )
    return fig, subplot_positions


def add_weekend_borders(
    fig: go.Figure,
    processed_months: list[tuple[pd.DataFrame, pd.DataFrame]],
    subplot_positions: list[tuple[int, int]],
) -> None:
    """Mark weekend cells with borders while preserving their cluster fill color."""
    for (month_data, _), (row, col) in zip(processed_months, subplot_positions):
        weekend_days = month_data[
            month_data["weekday_num"].isin(WEEKEND_WEEKDAY_NUMBERS)
        ]
        for _, day in weekend_days.iterrows():
            fig.add_shape(
                type="rect",
                x0=day["weekday_num"] - 0.5,
                x1=day["weekday_num"] + 0.5,
                y0=day["week_row"] - 0.5,
                y1=day["week_row"] + 0.5,
                line=dict(color="rgba(35, 35, 35, 0.75)", width=2),
                fillcolor="rgba(0, 0, 0, 0)",
                layer="above",
                row=row,
                col=col,
            )


### Full-Year Calendar Styling Helpers

In [105]:
def style_calendar_figure(fig: go.Figure, template: str) -> None:
    """Apply shared layout and axis styling to the full-year calendar."""
    fig.update_layout(
        title={
            "text": "2025 clusters calendar",
            "x": 0.5,
            "font_size": 20,
            "y": 0.99,
        },
        template=template,
        autosize=True,
        width=None,
        height=900,
        plot_bgcolor="white",
        margin={"l": 20, "r": 20, "t": 120, "b": 20},
    )
    fig.update_annotations(yshift=25)
    fig.update_xaxes(
        side="top",
        tickmode="array",
        tickvals=list(WEEKDAY_NUMBERS),
        ticktext=WEEKDAY_LABELS,
        showticklabels=True,
        ticks="",
        showline=False,
        showgrid=False,
        zeroline=False,
    )
    fig.update_yaxes(
        ticks="",
        showline=False,
        autorange="reversed",
        showgrid=False,
        zeroline=False,
        showticklabels=False,
    )


def build_cluster_calendar_figure(
    processed_months: list[tuple[pd.DataFrame, pd.DataFrame]],
    heatmaps: list[go.Heatmap],
    template: str,
) -> go.Figure:
    """Build the full-year cluster calendar figure."""
    fig, subplot_positions = create_calendar_subplot_figure(heatmaps)
    add_weekend_borders(fig, processed_months, subplot_positions)
    style_calendar_figure(fig, template)
    return fig


Build and display the calendar visualization.

In [106]:
fig = build_cluster_calendar_figure(
    processed_monthly_assignments, calendar_heatmaps, plotly_theme
)
fig.show()


## Cluster Weight Heatmap
This heatmap summarizes how many days from each month were assigned to each cluster.

### Data Preparation Helpers

In [107]:
month_cluster_counts = (
    calendar_assignments.assign(month=calendar_assignments.index.month_name())
    .groupby("month")["cluster"]
    .value_counts()
)

Pivot the data into a month-by-cluster matrix.

In [108]:
month_cluster_counts = month_cluster_counts.unstack(
    level="cluster", fill_value=0
).sort_index(axis=1)

Sort months in calendar order.

In [109]:
month_order: Final[list[str]] = [
    "January",
    "February",
    "March",
    "April",
    "May",
    "June",
    "July",
    "August",
    "September",
    "October",
    "November",
    "December",
]

month_cluster_counts.index = pd.CategoricalIndex(
    month_cluster_counts.index,
    categories=month_order,
    ordered=True,
)
month_cluster_counts.sort_index(inplace=True)

month_cluster_share_pct = (
    month_cluster_counts.div(month_cluster_counts.sum(axis=1), axis=0)
    .mul(100)
    .round(1)
)

### Plotting Code

In [110]:
# Column count changes when the annual cluster count changes.
cluster_count = len(month_cluster_share_pct.columns)

# Hide in-cell labels when the matrix becomes too dense to read.
show_cluster_heatmap_labels = month_cluster_share_pct.size <= 240

# Use more vertical space for larger cluster configurations.
cluster_heatmap_height = 520 if cluster_count <= 12 else 620

# Reserve extra bottom margin when cluster labels are rotated.
cluster_heatmap_bottom_margin = 70 if cluster_count <= 12 else 110

# Rotate cluster labels once there are too many to fit horizontally.
cluster_tick_angle = 0 if cluster_count <= 12 else -45

fig = px.imshow(
    month_cluster_share_pct,
    aspect="auto",
    labels={"x": "Cluster", "y": "Month", "color": "% of month"},
)

fig.update_traces(
    customdata=month_cluster_counts.to_numpy()[:, :, None],
    texttemplate="%{z:.1f}%" if show_cluster_heatmap_labels else "",
    hovertemplate=(
        "Month: %{y}<br>"
        "Cluster: %{x}<br>"
        "Share of month: %{z:.1f}%<br>"
        "Days assigned: %{customdata[0]:.0f}"
        "<extra></extra>"
    ),
)
fig.update_layout(
    title={
        "text": "Share of each month assigned to each cluster",
        "x": 0.5,
        "font_size": 20,
    },
    autosize=True,
    width=None,
    height=cluster_heatmap_height,
    margin={"l": 20, "r": 20, "t": 60, "b": cluster_heatmap_bottom_margin},
    coloraxis_showscale=False,
)
fig.update_xaxes(tickangle=cluster_tick_angle)
fig.show()

# Saving Results
TSAM outputs are pandas DataFrames and can be saved in any format supported by pandas.

[pandas-supported file formats](https://pandas.pydata.org/docs/user_guide/io.html)

This section saves the reconstructed 8760-hour dataset to a `.csv` file.

In [ ]:
# Create a directory where the results are going to be stored
output_path: Path = cur_location / "outputs"
os.makedirs(output_path, exist_ok=True)

# Save to disk in a specified path under file_name
file_name = "results.csv"
aggregation_result.reconstructed.to_csv(output_path / file_name)

print(f"Saved the data successfully in: {output_path}/{file_name}")

Saved the data successfully in: /Users/koval/dev/GDU/src/outputs/results.csv
